# Disco completo GOES-East (G19) · infrarrojo limpio C13 (10.3 µm)

Animación del **disco completo** con archivos reales
`OR_ABI-L2-CMIPF-M6C13_G19_*.nc` (NOAA/NESDIS, bucket público `noaa-goes19`).

- **Satélite y banda**: GOES-East (G19, en 75.2°O), banda **C13 (10.33 µm)**,
  temperatura de brillo en °C. Es la banda "Clean Infrared": ve nubes de día y de noche.
- **Encuadre**: todo lo que abarca el dataset (disco completo, sin zoom).
- **Escala**: percentiles automáticos sobre el primer cuadro, igual que `poster_goes.py`;
  paleta `termico` (frío = blanco, como en las imágenes IR de siempre).
- **Cadencia**: 10 min (modo 6); el nombre del archivo la trae escrita.
- **Vestuario**: el mismo de `poster_goes.py` (`draw_goes_poster` + `caption_lines`).


In [ ]:
# ==== configuración ====
from datetime import datetime
from pathlib import Path

# Raíz del proyecto, se corra el notebook desde notebooks/ o desde la raíz.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

T0       = datetime(2026, 10, 12, 21, 30)   # primer cuadro (UTC): 24 h hasta el ultimo dato
N_FRAMES = 144                              # 24 h a 10 min
PASO_MIN = 10                              # cadencia real de C13 en modo 6

SAT, BANDA = "G19", "C13"                  # GOES-East, IR limpio 10.3 µm
PALETA     = "ir_realce"                   # IR con realce de color
CREDITO    = "Graphic by @tu_usuario"      # tu handle: arriba a la derecha + marca de agua

DESCARGAR  = True                          # False = solo archivos locales; falla si falta un cuadro

# Escala de la barra: None = percentiles automáticos, igual que poster_goes.py.
VMIN, VMAX = None, None
ESCALA_PERCENTILES = (0.5, 99.5)
TOPE_PX = 2712                             # 1:1 con el dato (5424 px en 2 km -> 2712 en 4 km)

LADO_PULG, DPI = 19.3714, 140              # 2712 px por cuadro (1:1 con el dato)
GEOJSON = str(RAIZ / "datos" / "paraguay_gadm.json")             # borde destacado (GADM, 1893 vertices)
CENTRO_LON, CENTRO_LAT = -73.0, -18.0       # Orthographic girado: encuadre aprobado
CUADRO_MS, FUNDIDO_MS, FUNDIDO_PASOS = 80, 30, 1
HACER_GIF, HACER_MP4 = False, True          # a esta resolucion el GIF seria ~1 GB
CRF_MP4, PRESET_MP4 = 16, "slow"           # CRF mas bajo = mas calidad; preset lento comprime mejor


In [ ]:
# ==== datos y utilidades ====
import gc
import io, re, sys, urllib.request
from collections import defaultdict
from datetime import timedelta
from pathlib import Path

import numpy as np, xarray as xr, cartopy.crs as ccrs
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as MostrarImagen, display
from rich.progress import track

sys.path.insert(0, str(RAIZ / "lib"))
from poster_goes import (BG, PALETTES, caption_lines, draw_goes_poster,
                         to_display)

DATOS = Path(r"D:\Users\guill\dataset era5\goes_c13_g19")
DATOS.mkdir(exist_ok=True)
SALIDAS = Path(r"D:\Users\guill\dataset era5\salidas")
SALIDAS.mkdir(parents=True, exist_ok=True)

POR_HORA = defaultdict(list)


def claves_de_la_hora(t):
    """Claves S3 de la hora de `t` (una sola consulta por hora)."""
    hora = t.replace(minute=0, second=0, microsecond=0)
    if hora not in POR_HORA:
        pref = (f"ABI-L2-CMIPF/{hora.year}/{int(hora.strftime('%j')):03d}/"
                f"{hora.hour:02d}/")
        url = f"https://noaa-goes{SAT[1:]}.s3.amazonaws.com/?list-type=2&prefix={pref}"
        xml = urllib.request.urlopen(url, timeout=120).read().decode()
        POR_HORA[hora] = re.findall(r"<Key>([^<]+)</Key>", xml)
    return POR_HORA[hora]


PATRON = re.compile(r"OR_ABI-L2-CMIPF-M6(?P<banda>C\d+)_(?P<sat>G\d+)_s(?P<s>\d{14})_")


def clave_de(t):
    """Clave del archivo cuyo escaneo arranca en el slot de 10 min de `t`."""
    for clave in claves_de_la_hora(t):
        m = PATRON.search(clave)
        if not m or m["banda"] != BANDA or m["sat"] != SAT:
            continue
        inicio = datetime.strptime(m["s"][:13], "%Y%j%H%M%S")
        if (inicio.hour, inicio.minute) == (t.hour, t.minute):
            return clave
    raise LookupError(f"Sin archivo {SAT} {BANDA} para {t:%Y-%m-%d %H:%M} UTC")


def ruta_local(clave):
    """Si el .nc ya está en la carpeta del proyecto, se reutiliza sin copiarlo."""
    nombre = clave.rsplit("/", 1)[-1]
    for candidato in (Path(nombre), DATOS / nombre):
        if candidato.exists():
            return candidato
    return DATOS / nombre


def descargar(clave, permitir_red=True):
    destino = ruta_local(clave)
    if destino.exists() and destino.stat().st_size > 1_000_000:
        return destino
    if not permitir_red:
        raise SystemExit(f"Falta {destino.name} y DESCARGAR=False: "
                         f"poné DESCARGAR=True para bajarlo de S3.")
    url = f"https://noaa-goes{SAT[1:]}.s3.amazonaws.com/{clave}"
    with urllib.request.urlopen(url, timeout=300) as respuesta:
        with open(destino, "wb") as salida:
            while True:
                trozo = respuesta.read(1 << 20)
                if not trozo:
                    break
                salida.write(trozo)
    return destino


def cargar(nc, tope_px=None):
    """Disco completo listo para dibujar: campo en °C, grilla, proyección y rótulos."""
    tope_px = tope_px or TOPE_PX
    with xr.open_dataset(nc) as ds:
        cmi = ds["CMI"]
        paso = max(1, int(np.ceil(max(cmi.shape) / tope_px)))
        crudo = np.asarray(cmi.values[::paso, ::paso], dtype=np.float64)
        campo, unidad, tipo = to_display(crudo, cmi)
        gp = ds["goes_imager_projection"].attrs
        h = float(gp["perspective_point_height"])
        x = ds["x"].values[::paso] * h
        y = ds["y"].values[::paso] * h
        proj = ccrs.Geostationary(
            central_longitude=float(gp["longitude_of_projection_origin"]),
            satellite_height=h, sweep_axis=str(gp.get("sweep_angle_axis", "x")))
        rotulos = caption_lines(ds, cmi, tipo, "FULL DISK")
    return campo, x, y, proj, unidad, rotulos


SLOTS = [T0 + i * timedelta(minutes=PASO_MIN) for i in range(N_FRAMES)]
print(f"{len(SLOTS)} cuadros: {SLOTS[0]:%Y-%m-%d %H:%M} -> {SLOTS[-1]:%Y-%m-%d %H:%M} UTC")


In [ ]:
# ==== descarga (solo lo que falta; nada si DESCARGAR=False y todo está local) ====
rutas = []
faltan = []
for t in track(SLOTS, description="Descargando cuadros"):
    try:
        ruta = descargar(clave_de(t), DESCARGAR)
    except LookupError:
        print(f"  salto {t:%Y-%m-%d %H:%M} UTC (hueco en NOAA)")
        faltan.append(t)
        continue
    rutas.append(ruta)
    print(f"{t:%Y-%m-%d %H:%M} UTC  {ruta.name}  {ruta.stat().st_size / 1e6:.1f} MB")
print(f"listos: {len(rutas)}/{len(SLOTS)} | huecos: {len(faltan)}")


In [ ]:
# ==== control: rótulos del primer cuadro ====
campo, x, y, proj, unidad, rotulos = cargar(rutas[0])
titulo, subtitulo, fuente, datos = rotulos
if VMIN is None or VMAX is None:
    lo, hi = np.nanpercentile(campo, ESCALA_PERCENTILES)
    VMIN = VMIN if VMIN is not None else float(lo)
    VMAX = VMAX if VMAX is not None else float(hi)
    print(f"escala automática {ESCALA_PERCENTILES}: {VMIN:.2f} a {VMAX:.2f} {unidad}")
print(f"disco: {campo.shape} celdas")
print(titulo)
print(subtitulo)
print(fuente)
print(datos)
assert unidad == "°C"
assert titulo == "OCTOBER 2026"
assert subtitulo == "G19 GOES-East\nTEMPERATURA DE BRILLO · BANDA 13 (10.3 µm)"
assert re.match(r"DATA: G19 \| \d{4}-\d{2}-\d{2}T\d{2}:\d{2} UTC \| MODO Mode 6", datos), datos
assert fuente == "SOURCE: NOAA/NESDIS G19 ABI L2 CMIP | FULL DISK"
print("\nOK: rótulos verificados.")


In [ ]:
# ==== cuadro fijo (PNG) del primer instante ====
fig, ax, _ = draw_goes_poster(
    campo, x=x, y=y, proj=proj, title=titulo, subtitle=subtitulo, unit_label=unidad,
    src_line=fuente, data_line=datos, credit=CREDITO,
    cmap=PALETTES[PALETA], vmin=VMIN, vmax=VMAX,
    highlight_geojson=GEOJSON, globe_center=(CENTRO_LON, CENTRO_LAT))
png = SALIDAS / f"poster_globo_{SAT}_{T0:%Y-%m-%dT%H%M}.png"
fig.savefig(png, facecolor=BG)
plt.close(fig)
print("PNG:", png)
display(MostrarImagen(filename=str(png)))


In [ ]:
# ==== render de todos los cuadros ====
# Si ya hay un cuadro por slot en frames_dir, se reutilizan (re-ejecutable sin re-renderizar).
frames_dir = SALIDAS / f"frames_{SAT}_{T0:%Y-%m-%dT%H%M}_{int(round(LADO_PULG * DPI))}px"
frames_dir.mkdir(parents=True, exist_ok=True)
archivos = sorted(frames_dir.glob("cuadro_*.png"))

if len(archivos) == len(rutas):
    print(f"{len(archivos)} cuadros ya renderizados en {frames_dir} (se saltea el render)")
else:
    archivos = []
    for i, ruta in enumerate(track(rutas, description="Renderizando"), 1):
        campo_i, x_i, y_i, proj_i, unidad_i, rot_i = cargar(ruta)
        fig, ax, _ = draw_goes_poster(
            campo_i, x=x_i, y=y_i, proj=proj_i, title=rot_i[0], subtitle=rot_i[1],
            unit_label=unidad_i, src_line=rot_i[2], data_line=rot_i[3],
            credit=CREDITO, cmap=PALETTES[PALETA], vmin=VMIN, vmax=VMAX,
            size=LADO_PULG, dpi=DPI, highlight_geojson=GEOJSON,
            globe_center=(CENTRO_LON, CENTRO_LAT))
        fp = frames_dir / f"cuadro_{i:03d}.png"
        fig.savefig(fp, format="png", facecolor=BG)
        plt.close(fig)
        archivos.append(fp)
        del campo_i, x_i, y_i, fig, fp
        gc.collect()
        print(f"  cuadro {i}/{len(rutas)}  {rot_i[3]}")

print(f"{len(archivos)} cuadros en {frames_dir}")


In [ ]:
# ==== GIF con fundido entre cuadros ====
def cuadros_con_fundido(paths, pasos, ms_cuadro, ms_fundido):
    """Genera (imagen P, duracion_ms) leyendo de a dos cuadros por vez (poca RAM)."""
    prev = Image.open(paths[0]).convert("RGB")
    yield prev.convert("P", palette=Image.ADAPTIVE), ms_cuadro
    for q in paths[1:]:
        cur = Image.open(q).convert("RGB")
        if pasos:
            a0 = np.asarray(prev, dtype=np.float32)
            a1 = np.asarray(cur, dtype=np.float32)
            for k in range(1, pasos + 1):
                alfa = k / (pasos + 1)
                mezcla = Image.fromarray(
                    ((1 - alfa) * a0 + alfa * a1).astype(np.uint8))
                yield mezcla.convert("P", palette=Image.ADAPTIVE), ms_fundido
        yield cur.convert("P", palette=Image.ADAPTIVE), ms_cuadro
        del prev
        prev = cur
        gc.collect()


if HACER_GIF:
    fotogramas = list(cuadros_con_fundido(
        sorted(archivos), FUNDIDO_PASOS, CUADRO_MS, FUNDIDO_MS))
    gif = SALIDAS / f"animacion_globo_{SAT}_{T0:%Y-%m-%dT%H%M}.gif"
    fotogramas[0][0].save(
        gif, save_all=True,
        append_images=[im for im, _ in fotogramas[1:]],
        duration=[ms for _, ms in fotogramas], loop=0, optimize=True)
    print(f"GIF: {gif}  ({len(fotogramas)} fotogramas, {gif.stat().st_size / 1e6:.1f} MB)")
    display(MostrarImagen(filename=str(gif)))
else:
    print("GIF desactivado (HACER_GIF=False): mirá el MP4 de la celda siguiente")


In [ ]:
# ==== MP4: misma animación en mucho menos peso y mejor calidad ====
# ffmpeg viene con imageio-ffmpeg, no hace falta que esté en el PATH.
import subprocess

import imageio_ffmpeg

if HACER_MP4:
    px = int(round(LADO_PULG * DPI))
    mp4 = SALIDAS / f"animacion_globo_{SAT}_{T0:%Y-%m-%dT%H%M}_{px}px.mp4"
    proc = subprocess.run([
        imageio_ffmpeg.get_ffmpeg_exe(), "-y",
        "-framerate", f"{1000 / CUADRO_MS:.6f}",   # cada cuadro real dura CUADRO_MS
        "-start_number", "1",
        "-i", str(frames_dir / "cuadro_%03d.png"),
        # x264 con yuv420p exige lados pares: el pad ajusta 1 px sin remuestrear.
        "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
        "-c:v", "libx264", "-preset", PRESET_MP4, "-pix_fmt", "yuv420p",
        "-crf", str(CRF_MP4),
        "-movflags", "+faststart", str(mp4),
    ], capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode:
        print(proc.stderr[-1500:])          # que se vea el motivo si falla
        raise RuntimeError(f"ffmpeg falló con código {proc.returncode}")
    print(f"MP4: {mp4}  ({mp4.stat().st_size / 1e6:.1f} MB, {px}x{px})")


## Ideas y variantes

- **Otra ventana**: cambiá `T0` y `N_FRAMES`; la clave S3 se arma sola a partir del
  slot de 10 minutos (no hace falta escribir el nombre del archivo).
- **Sin descargar**: `DESCARGAR = False` usa solo los .nc que ya estén en la carpeta
  (o en `goes_c13_g19/`); si falta un cuadro, avisa cuál en vez de bajarlo.
- **Zoom regional**: ver `animacion_paraguay_g19.ipynb` (misma estructura, recorte a la
  zona `py` a resolución completa).
- **Escala fija**: poné `VMIN`/`VMAX` en °C para no depender del primer cuadro.
- **Sin fundido**: `FUNDIDO_PASOS = 0` deja la animación a saltos limpios de 10 min.
- **MP4**: se genera con el ffmpeg que trae `imageio-ffmpeg` (~7x menos peso que
  el GIF y sin dithering). `HACER_GIF=False` saltea el GIF, que a 1176 px pesa mucho.
- **Crédito**: `CREDITO` alimenta el texto de arriba a la derecha y la marca de agua
  central (esa se dibuja con las letras separadas).
